# Breast Cancer Classification — Training and Evaluation

This notebook trains a preprocessing + Logistic Regression pipeline and saves `model.joblib`.

**Dataset:** For a Kaggle-sourced run, download the Kaggle *Breast Cancer Wisconsin (Diagnostic)* CSV and upload it in the first cell. The common Kaggle file is `data.csv` and contains a `diagnosis` column (`M`/`B`). If you don't upload a CSV, the notebook uses scikit-learn's bundled copy of the same canonical UCI dataset so the notebook can run immediately.

Educational demonstration only; not for medical diagnosis.

In [ ]:
# Install dependencies in Google Colab
!pip -q install scikit-learn pandas joblib

In [ ]:
from pathlib import Path
import pandas as pd
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

## 1. Load the dataset

For a Kaggle dataset run, use the upload button and select the downloaded CSV. Download the dataset from Kaggle in your browser first; Kaggle API credentials are not required for this manual-upload workflow.

In [ ]:
# Optional: upload a Kaggle CSV in Colab.
# If you skip upload, the canonical scikit-learn/UCI copy is used.
USE_KAGGLE_CSV = False

if USE_KAGGLE_CSV:
    from google.colab import files
    uploaded = files.upload()
    csv_name = next(name for name in uploaded if name.lower().endswith(".csv"))
    raw = pd.read_csv(csv_name)
    print("Loaded:", csv_name, "shape:", raw.shape)
    display(raw.head())
else:
    dataset = load_breast_cancer(as_frame=True)
    X = dataset.data.copy()
    y = dataset.target.copy()
    class_names = list(dataset.target_names)
    print("Loaded bundled UCI copy:", X.shape)
    display(X.head())

## 2. Clean and align the Kaggle columns

The Kaggle CSV normally contains `id`, `diagnosis`, and 30 measurement columns. The mapping below standardizes its column names to the feature names used by the API.

In [ ]:
if USE_KAGGLE_CSV:
    raw = raw.drop(columns=[c for c in ["id", "Unnamed: 32"] if c in raw.columns])

    kaggle_to_model = {
        "radius_mean": "mean radius",
        "texture_mean": "mean texture",
        "perimeter_mean": "mean perimeter",
        "area_mean": "mean area",
        "smoothness_mean": "mean smoothness",
        "compactness_mean": "mean compactness",
        "concavity_mean": "mean concavity",
        "concave points_mean": "mean concave points",
        "symmetry_mean": "mean symmetry",
        "fractal_dimension_mean": "mean fractal dimension",
        "radius_se": "radius error",
        "texture_se": "texture error",
        "perimeter_se": "perimeter error",
        "area_se": "area error",
        "smoothness_se": "smoothness error",
        "compactness_se": "compactness error",
        "concavity_se": "concavity error",
        "concave points_se": "concave points error",
        "symmetry_se": "symmetry error",
        "fractal_dimension_se": "fractal dimension error",
        "radius_worst": "worst radius",
        "texture_worst": "worst texture",
        "perimeter_worst": "worst perimeter",
        "area_worst": "worst area",
        "smoothness_worst": "worst smoothness",
        "compactness_worst": "worst compactness",
        "concavity_worst": "worst concavity",
        "concave points_worst": "worst concave points",
        "symmetry_worst": "worst symmetry",
        "fractal_dimension_worst": "worst fractal dimension",
    }
    if "diagnosis" not in raw.columns:
        raise ValueError("Expected a 'diagnosis' column in the Kaggle CSV.")
    missing = sorted(set(kaggle_to_model) - set(raw.columns))
    if missing:
        raise ValueError(f"CSV is missing expected measurement columns: {missing}")

    X = raw.rename(columns=kaggle_to_model)[list(kaggle_to_model.values())].copy()
    # The API uses the scikit-learn/UCI target convention: 0=malignant, 1=benign.
    y = raw["diagnosis"].map({"M": 0, "B": 1})
    if y.isna().any():
        raise ValueError("Diagnosis must contain only 'M' or 'B'.")
    class_names = ["malignant", "benign"]

print("Feature matrix:", X.shape)
print("Target distribution:", y.value_counts().to_dict())
assert X.shape[1] == 30, "Expected 30 measurement features."
assert not X.isna().any().any(), "Missing feature values detected."
assert y.notna().all()

## 3. Split, train and evaluate

Keep the test split separate from model training. We use a stratified 80/20 split and report several metrics.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=3000, random_state=42)),
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
}
print("Metrics:", metrics)
print("\nConfusion matrix (rows=true, columns=predicted):\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(
    y_test, y_pred, target_names=class_names, zero_division=0
))

## 4. Save the model and its parameters

The artifact stores the fitted pipeline, exact feature order, class names, metrics and dataset metadata. The API loads this file at startup.

In [ ]:
artifact = {
    "model": model,
    "feature_names": list(X.columns),
    "class_names": class_names,
    "metrics": metrics,
    "dataset": "Breast Cancer Wisconsin (Diagnostic)",
    "random_state": 42,
}
output_path = Path("model.joblib")
joblib.dump(artifact, output_path)
print(f"Saved {output_path.resolve()} ({output_path.stat().st_size:,} bytes)")

# Reload it to verify serialization
loaded = joblib.load(output_path)
assert loaded["feature_names"] == list(X.columns)
print("Reload successful. Stored metrics:", loaded["metrics"])

## 5. Download the artifact for VS Code

Download `model.joblib` and place it in the local project's `app/` directory.

In [ ]:
from google.colab import files
files.download("model.joblib")